# EYES-DEFY-ANEMIA -- Fine-tune Pilot -- All 6 Models, One Notebook

Trains all 6 models in the partial fine-tuning pilot programme (`classification/.project_memory/
13_finetune_pilot_programme.md`) in a single notebook run: **ConvNeXt-Base, CoAtNet-3, EfficientNet-B3**
(batch 1) and **ConvNeXt-Large, MaxViT-T, RegNetY-16GF** (batch 2), then builds one cross-model comparison
across all 6. Cells are ordered cheapest-checkpoint-first (EfficientNet-B3 43MB -> MaxViT-T 123MB ->
RegNetY-16GF 323MB -> ConvNeXt-Base 350MB -> CoAtNet-3 655MB -> ConvNeXt-Large 785MB), same convention
already used for this project's other multi-model sweep notebooks, so a mid-run interruption loses as
little progress as possible -- `sync_outputs()` runs after every training cell, not just at the end.

**Note:** ConvNeXt-Base and CoAtNet-3 already have real, Kaggle-executed results from their own individual
notebooks (`finetune-convnext-base.ipynb`, `finetune-coatnet-3.ipynb`) -- running them again here reproduces
that same result under a fresh checkpoint filename, `_v2`-style, consistent with this project's existing
convention (distinct filenames rather than silently overwriting).

**Data, all under the `manivafapour33` Kaggle account:**
1. **`processed-dataset-clean`** -- real VAL/TEST images plus `splits.csv`/`extraction_log.csv`.
2. **`checkpoints`** -- needs a new version with all 6 source `.pth` files (a ready-to-upload
   `checkpoints_all6.zip` was prepared locally at
   `classification/new_way/Output/version1/checkpoints/checkpoints_all6.zip`, 2279 MB, integrity-verified).

TRAIN reads the offline-balanced + online-augmented data (`Offline_data_augmentation/`, arrives via
`git clone`, no separate dataset needed).

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic only -- run this BEFORE filling in the TODO paths in the Data section below.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

In [ ]:
# optuna is required even though this notebook never runs a search -- datapreparepipeline/
# trainer_engine.py (reused for compute_metrics/evaluate) imports it unconditionally at module level.
# timm is required for CoAtNet-3 -- it does not exist in torchvision at all. The other 5 models are
# all torchvision-native.
!pip install -q optuna albumentations timm

## Data

Two things need to be attached as Kaggle datasets for this notebook to run, both under **`manivafapour33`**:

1. **`processed-dataset-clean`** -- provides the real VAL/TEST images plus `splits.csv`/`extraction_log.csv`.
2. **`checkpoints-all6`**, containing all 6 source checkpoints (`checkpoints_all6.zip`, see the intro above;
   Kaggle mounts a dataset titled `checkpoints_all6` as `checkpoints-all6` -- underscores become hyphens) --
   `best_convnext_base_palpebral_new_way.pth`, `best_coatnet_3_palpebral_new_way.pth`,
   `best_efficientnet_b3_forniceal_palpebral_new_way.pth`, `best_convnext_large_palpebral_new_way.pth`,
   `best_maxvit_t_palpebral_new_way.pth`, `best_regnet_y_16gf_palpebral_new_way.pth`.

Check the `/kaggle/input` listing above and fill in both TODO paths below before running.

In [ ]:
import shutil
from pathlib import Path

# TODO: verify against the /kaggle/input listing cell above before running.
PROCESSED_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in PROCESSED_SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

In [ ]:
# TODO: verify against the /kaggle/input listing cell above before running -- this is the
# checkpoint dataset (see markdown above), not the same as PROCESSED_SRC_DIR. All 6 required
# checkpoints must be present in this dataset. Note: Kaggle slugifies dataset titles with
# underscores into hyphens -- a dataset titled "checkpoints_all6" mounts as "checkpoints-all6",
# confirmed directly against a live Kaggle session, not assumed.
CHECKPOINT_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/checkpoints-all6")
CHECKPOINT_DST_DIR = Path("classification/new_way/Output/version1/checkpoints")
CHECKPOINT_DST_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for item in CHECKPOINT_SRC_DIR.rglob("*.pth"):
    shutil.copy2(item, CHECKPOINT_DST_DIR / item.name)
    copied += 1
    print(f"  copied {item.name} ({item.stat().st_size / 1e6:.1f} MB)")
print(f"\n{copied} checkpoint file(s) staged to {CHECKPOINT_DST_DIR}")

In [ ]:
# Fails loudly here, not deep inside training, if either data source above is missing/misconfigured.
manifest_path = Path("classification/new_way/Offline_data_augmentation/manifest.csv")
assert manifest_path.exists(), (
    f"{manifest_path} not found -- Offline_data_augmentation/ should have arrived via git clone. "
    "Was it actually committed and pushed?"
)

required_checkpoints = [
    "best_efficientnet_b3_forniceal_palpebral_new_way.pth",
    "best_maxvit_t_palpebral_new_way.pth",
    "best_regnet_y_16gf_palpebral_new_way.pth",
    "best_convnext_base_palpebral_new_way.pth",
    "best_coatnet_3_palpebral_new_way.pth",
    "best_convnext_large_palpebral_new_way.pth",
]
for name in required_checkpoints:
    p = CHECKPOINT_DST_DIR / name
    assert p.exists(), (
        f"{p} not found -- fix CHECKPOINT_SRC_DIR above to point at the real "
        "checkpoint-dataset mount path (check the /kaggle/input listing cell), and confirm "
        "the dataset actually contains all 6 required checkpoints."
    )
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")
print(f"\nAll data sources present: {manifest_path}, and all {len(required_checkpoints)} required checkpoints.")

## Sanity checks

Builds each fine-tune model for real (loads its checkpoint, applies the freeze/unfreeze split) and confirms
the trainable-parameter count matches what was verified locally, before any real training starts.

In [ ]:
import sys
sys.path.insert(0, "classification/new_way/Fine_tune")

In [ ]:
import finetune_engine_efficientnet_b3 as fe_eff

model = fe_eff.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B3 trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "2,246,497", (
    f"Expected 2,246,497 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_maxvit_t as fe_mt

model = fe_mt.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"MaxViT-T trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "8,962,465", (
    f"Expected 8,962,465 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_regnet_y_16gf as fe_rn

model = fe_rn.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"RegNetY-16GF trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "9,153,649", (
    f"Expected 9,153,649 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine as fe_cb

model = fe_cb.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"ConvNeXt-Base trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "8,449,025", (
    f"Expected 8,449,025 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_coatnet3 as fe_ca

model = fe_ca.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"CoAtNet-3 trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "9,457,585", (
    f"Expected 9,457,585 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

In [ ]:
import finetune_engine_convnext_large as fe_cl

model = fe_cl.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"ConvNeXt-Large trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "18,964,993", (
    f"Expected 18,964,993 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

## Output syncing

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate Fine_tune/Output/{checkpoints,logs,plots}/ into a single top-level
    /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/all6_finetune_results.zip. Called after every training cell below, not just
    once at the end -- this is the longest-running notebook in the programme (6 models), so
    protecting against a mid-run interruption matters more here than anywhere else."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/new_way/Fine_tune/Output") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/all6_finetune_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

## Training -- EfficientNet-B3 (43MB checkpoint -- cheapest)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_efficientnet_b3_forniceal.py
sync_outputs()

## Training -- MaxViT-T (123MB checkpoint)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_maxvit_t_palpebral.py
sync_outputs()

## Training -- RegNetY-16GF (323MB checkpoint)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_regnet_y_16gf_palpebral.py
sync_outputs()

## Training -- ConvNeXt-Base (350MB checkpoint)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_convnext_base_palpebral.py
sync_outputs()

## Training -- CoAtNet-3 (655MB checkpoint)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_coatnet3_palpebral.py
sync_outputs()

## Training -- ConvNeXt-Large (785MB checkpoint -- priciest, queued last)

In [ ]:
!python classification/new_way/Fine_tune/train_finetune_convnext_large_palpebral.py
sync_outputs()

## Cross-model comparison (all 6 models)

Loads all 6 models' `*_history.json` (all just written by the training cells above, in this same run) and
builds the full-programme comparison -- 3 plots (val F1, test F1/Accuracy/AUC, India/Italy AUC gap) plus a
summary table, using the same `plot_cross_model_comparison()` already verified against batch 1's results.

In [ ]:
import finetune_common as fc

FINE_TUNE_LOGS_DIR = Path("classification/new_way/Fine_tune/Output/logs")
FINE_TUNE_PLOTS_DIR = Path("classification/new_way/Fine_tune/Output/plots")

ALL_SIX_MODEL_NAMES = [
    "convnext_base_palpebral_new_way_finetune_block3_v2",
    "coatnet_3_palpebral_new_way_finetune_attn",
    "efficientnet_b3_forniceal_palpebral_new_way_finetune_block_v2",
    "convnext_large_palpebral_new_way_finetune_block3",
    "maxvit_t_palpebral_new_way_finetune_lastlayer",
    "regnet_y_16gf_palpebral_new_way_finetune_finalconv",
]

histories = {name: fc.load_finetune_history(FINE_TUNE_LOGS_DIR, name) for name in ALL_SIX_MODEL_NAMES}
comparison_paths = fc.plot_cross_model_comparison(histories, FINE_TUNE_PLOTS_DIR, FINE_TUNE_LOGS_DIR)
print(comparison_paths)

sync_outputs()

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` and zipped to `/kaggle/working/all6_finetune_results.zip`.
Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All.

The headline number for each model is in its own `{model_name}_history.json`'s `best_val_f1` and
`test_metrics` -- compare against this notebook's own printed before/after blocks per model, and against the
6-model cross-model comparison plots/table (`cross_model_val_f1_comparison.png`,
`cross_model_test_metrics_comparison.png`, `cross_model_auc_gap_comparison.png`,
`cross_model_comparison_table.md`).

In [ ]:
print("Final contents of /kaggle/working/outputs:")
for f in sorted(Path("/kaggle/working/outputs").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/kaggle/working/outputs')}  ({f.stat().st_size / 1e6:.2f} MB)")

zip_path = Path("/kaggle/working/all6_finetune_results.zip")
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")